# Agente Speech-to-Speech open source con MiniCPM-o 4.5

MiniCPM-o 4.5 es un modelo multimodal abierto para conversación por voz. Recibe audio y puede generar texto y audio localmente:

```text
WAV del usuario -> MiniCPM-o 4.5 -> respuesta textual + respuesta hablada
```

Este notebook no utiliza APIs de pago ni claves. Usa la inferencia oficial mediante Transformers y procesa el audio en el equipo. El modelo tiene aproximadamente 9B parámetros: la variante PyTorch requiere una GPU con memoria elevada; para equipos modestos deben estudiarse las variantes cuantizadas GGUF/AWQ mediante los runtimes documentados por el proyecto.

## 1. Instalacion

Instala las versiones recomendadas por la documentación oficial en el entorno que usará Jupyter. Para TTS y conversación hablada se necesita `minicpmo-utils[all]`. Esta celda es opcional para evitar instalaciones automáticas inesperadas.

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_DEPENDENCIES = False
PACKAGES = [
    "setuptools<70",
    "transformers==4.51.0",
    "accelerate",
    "torch>=2.3.0,<=2.8.0",
    "torchaudio<=2.8.0",
    "minicpmo-utils[all]>=1.0.5",
    "librosa",
    "soundfile",
]

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", *PACKAGES], check=True)
    print("Instalación terminada. Reinicia el kernel antes de continuar.")
else:
    print("Instalacion desactivada. Cambia INSTALL_DEPENDENCIES a True solo si necesitas instalar.")
    missing = [
        package for module, package in {
            "torch": "torch",
            "transformers": "transformers",
            "accelerate": "accelerate",
            "librosa": "librosa",
            "soundfile": "soundfile",
            "minicpmo": "minicpmo-utils[all]",
        }.items()
        if importlib.util.find_spec(module) is None
    ]
    print("Paquetes ausentes:", ", ".join(missing) or "ninguno")

Instalacion desactivada. Cambia INSTALL_DEPENDENCIES a True solo si necesitas instalar.
Paquetes ausentes: minicpmo-utils[all]


In [2]:
import platform
import torch

IN_COLAB = "google.colab" in sys.modules
CUDA_AVAILABLE = torch.cuda.is_available()
print(f"Python: {sys.version.split()[0]}")
print(f"Ejecutable: {sys.executable}")
print(f"Sistema: {platform.platform()}")
print(f"Entorno: {'Google Colab' if IN_COLAB else 'local'}")
print(f"PyTorch: {torch.__version__}")

if CUDA_AVAILABLE:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"CUDA: disponible | GPU: {GPU_NAME} | VRAM: {GPU_MEMORY_GB:.1f} GB")
else:
    GPU_NAME = "CPU"
    GPU_MEMORY_GB = 0.0
    print("CUDA: no disponible. La inferencia PyTorch oficial de MiniCPM-o 4.5 necesita preferiblemente una GPU Nvidia.")

if not CUDA_AVAILABLE:
    print("Para CPU, revisa las variantes GGUF y el soporte de llama.cpp/Ollama del repositorio oficial.")

DTYPE = torch.bfloat16 if CUDA_AVAILABLE and torch.cuda.is_bf16_supported() else torch.float16
print(f"Tipo numerico previsto: {DTYPE}")

Python: 3.12.10
Ejecutable: c:\Users\oitav\Documents\VIU\TFM\voice-agents\voiceagent\Scripts\python.exe
Sistema: Windows-11-10.0.26200-SP0
Entorno: local
PyTorch: 2.8.0+cpu
CUDA: no disponible. La inferencia PyTorch oficial de MiniCPM-o 4.5 necesita preferiblemente una GPU Nvidia.
Para CPU, revisa las variantes GGUF y el soporte de llama.cpp/Ollama del repositorio oficial.
Tipo numerico previsto: torch.float16


## 2. Configuracion y audio de entrada

La API oficial espera el audio como un array mono a 16 kHz. Se usa un WAV corto para que la prueba sea repetible y no dependa de un micrófono en tiempo real.

In [4]:
import librosa
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

MODEL_NAME = "openbmb/MiniCPM-o-4_5"
LANGUAGE = "en"  # MiniCPM-o 4.5 documenta conversación hablada en inglés y chino.
SYSTEM_PROMPT = "Please answer briefly and naturally."
USER_PROMPT = "Listen to the user and answer the question."
WORKSPACE_DIR = Path.cwd()
INPUT_AUDIO = WORKSPACE_DIR / "input.wav"
OUTPUT_AUDIO = WORKSPACE_DIR / "output_minicpmo.wav"
INPUT_SAMPLE_RATE = 16_000
OUTPUT_SAMPLE_RATE = 24_000

if not INPUT_AUDIO.exists():
    test_audio_candidates = [
        WORKSPACE_DIR.parent / "models" / "audio" / "usuario.wav",
        WORKSPACE_DIR.parent / "models" / "audio" / "prueba_kernel.wav",
        WORKSPACE_DIR.parent / "models" / "piper" / "prueba.wav",
    ]
    INPUT_AUDIO = next(
        (candidate for candidate in test_audio_candidates if candidate.exists()),
        INPUT_AUDIO,
    )

if IN_COLAB and not INPUT_AUDIO.exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No se ha seleccionado un audio.")
    INPUT_AUDIO = Path(next(iter(uploaded)))

if not INPUT_AUDIO.exists():
    raise FileNotFoundError(
        f"No existe el audio: {INPUT_AUDIO.resolve()}. "
        "Configura INPUT_AUDIO con la ruta de un WAV, FLAC u OGG existente."
    )

audio_info = sf.info(INPUT_AUDIO)
if audio_info.duration <= 0 or audio_info.duration > 30:
    raise ValueError("Usa un audio con una duracion entre 0 y 30 segundos.")

audio_input, _ = librosa.load(INPUT_AUDIO, sr=INPUT_SAMPLE_RATE, mono=True)
print(f"Audio: {INPUT_AUDIO.resolve()}")
print(f"Duracion: {len(audio_input) / INPUT_SAMPLE_RATE:.2f} s")
display(Audio(audio_input, rate=INPUT_SAMPLE_RATE))

Audio: C:\Users\oitav\Documents\VIU\TFM\voice-agents\models\audio\usuario.wav
Duracion: 5.00 s


In [6]:
import time
from importlib.metadata import version
from transformers import AutoModel

print(f"Transformers: {version('transformers')}")
if CUDA_AVAILABLE:
    MODEL_DEVICE = "cuda"
    LOAD_DTYPE = DTYPE
    print("Cargando MiniCPM-o 4.5 en CUDA; la descarga de pesos puede ser grande.")
else:
    MODEL_DEVICE = "cpu"
    LOAD_DTYPE = torch.float32
    print(
        "Cargando MiniCPM-o 4.5 en CPU. Esta prueba local puede tardar mucho, "
        "consumir mucha RAM o no completarse; la ruta recomendada sigue siendo CUDA."
    )

load_started = time.perf_counter()
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    attn_implementation="sdpa",
    torch_dtype=LOAD_DTYPE,
    init_vision=False,
    init_audio=True,
    init_tts=True,
)
model.eval().to(MODEL_DEVICE)
model.init_tts()
load_seconds = time.perf_counter() - load_started
print(f"Modelo cargado en {load_seconds:.2f} s")

Transformers: 4.57.3
Cargando MiniCPM-o 4.5 en CPU. Esta prueba local puede tardar mucho, consumir mucha RAM o no completarse; la ruta recomendada sigue siendo CUDA.


config.json: 0.00B [00:00, ?B/s]

c:\Users\oitav\Documents\VIU\TFM\voice-agents\voiceagent\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\oitav\.cache\huggingface\hub\models--openbmb--MiniCPM-o-4_5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


configuration_minicpmo.py: 0.00B [00:00, ?B/s]

modeling_navit_siglip.py: 0.00B [00:00, ?B/s]

...

processing_minicpmo.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/openbmb/MiniCPM-o-4_5:
- processing_minicpmo.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


utils.py: 0.00B [00:00, ?B/s]

Encountered exception while importing minicpmo: No module named 'minicpmo'


ImportError: This modeling file requires the following packages that were not found in your environment: minicpmo. Run `pip install minicpmo`

## 3. Inferencia S2S por turnos

Esta llamada sigue el ejemplo oficial de conversación de audio. El prompt del sistema define el comportamiento y el mensaje del usuario contiene el audio y la instrucción. `generate_audio=True` solicita la respuesta hablada.

In [ ]:
conversation = [
    {
        "role": "system",
        "content": [SYSTEM_PROMPT],
    },
    {
        "role": "user",
        "content": [USER_PROMPT, audio_input],
    },
]

inference_started = time.perf_counter()
response = model.chat(
    msgs=conversation,
    do_sample=True,
    temperature=0.3,
    max_new_tokens=512,
    use_tts_template=True,
    enable_thinking=False,
    omni_mode=True,
    generate_audio=True,
    output_audio_path=str(OUTPUT_AUDIO),
)
inference_seconds = time.perf_counter() - inference_started

print("Respuesta textual:")
print(response)
print(f"Tiempo de inferencia: {inference_seconds:.2f} s")
print(f"Audio generado: {OUTPUT_AUDIO.resolve()}")
display(Audio(filename=str(OUTPUT_AUDIO)))

In [ ]:
result = {
    "model": MODEL_NAME,
    "language": LANGUAGE,
    "device": "cuda",
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "load_seconds": load_seconds,
    "inference_seconds": inference_seconds,
    "input_duration_seconds": len(audio_input) / INPUT_SAMPLE_RATE,
    "output_path": str(OUTPUT_AUDIO),
}

result

## 4. Notas para el TFM

- MiniCPM-o 4.5 es open source/open weights bajo Apache-2.0 según el repositorio y la model card oficiales. Revisa la revisión concreta antes de distribuir resultados.
- La documentación oficial declara conversación hablada bilingüe en inglés y chino; no debe evaluarse como modelo español sin una prueba lingüística específica.
- La inferencia PyTorch de esta primera prueba no es todavía full-duplex realtime: usa un turno completo en WAV.
- Para tiempo real se debe usar `streaming_prefill` y `streaming_generate`, junto con captura de audio, VAD y reproducción por fragmentos.
- Fuentes: https://github.com/OpenBMB/MiniCPM-V, https://huggingface.co/openbmb/MiniCPM-o-4_5 y https://github.com/OpenSQZ/MiniCPM-V-CookBook.